<a href="https://colab.research.google.com/github/sv650s/amazon-review-classification/blob/master/5.1-LSTM-misclassification-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Miscategorized Reviews by LSTM

In previous notebooks, we filtered and trained LSTM network with 16 nodes based on original sample size of 1 million samples

Variations and links to notebooks:


* [1 and 5 star ratings](deep_learning/6.6.3-LSTMB16_15-GloVe-1m-prototype.ipynb)
* [1, 4, 5 star ratings](deep_learning/6.7.3-LSTMB16_145-GloVe-1m-prototype.ipynb)
* [1, 3, 4, 5 star ratings](deep_learning/6.8.3-LSTMB16_1345-GloVe-1m-prototype.ipynb)
* [All star ratings](deep_learning/6.5.3-LSTMB16-GloVe-1m-prototype.ipynb)

In this notebook, we will look at the model performance as well as look at specific reviews that were miscategorized

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
DRIVE_DIR = "drive/My Drive/Springboard/capstone"
sys.path.append(DRIVE_DIR)


%tensorflow_version 2.x


import tensorflow as tf
# checl to make sure we are using GPU here
tf.test.gpu_device_name()

In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Embedding, \
    SpatialDropout1D, Flatten, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.utils import model_to_dot
from tensorflow.keras.initializers import Constant
from tensorflow.keras.models import load_model



from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight


import pandas as pd
import numpy as np
from IPython.display import SVG
import pickle
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import os
import logging
import json
from pprint import pprint


import util.dict_util as du
import util.plot_util as pu
import util.file_util as fu
import util.keras_util as ku
import util.report_util as ru

import random

# fix random seeds
tf.compat.v1.reset_default_graph()
tf.compat.v1.set_random_seed(1)
random.seed(1)
np.random.seed(1)

logging.basicConfig(level=logging.ERROR)

%matplotlib inline
sns.set(font_scale=1.5)


DATE_FORMAT = '%Y-%m-%d'
TIME_FORMAT = '%Y-%m-%d %H:%M:%S'
LABEL_COLUMN = "star_rating"
REVIEW_COLUMN = "review_body"


DEBUG = False

pd.set_option("max_colwidth", 150)
REPORT_FILE = f'{DRIVE_DIR}/reports/glove_embedding-dl_prototype-report.csv'



Read in report file

In [ ]:
report = pd.read_csv(REPORT_FILE, quotechar="'")
report["train_examples"] = report.train_examples.astype(int)
# 1-star and 5-star only
lstm_15 = report[(report.model_name == "LSTMB16_15") & (report.train_examples == 505410)].copy().reset_index()
# 1-star, 4-star, and 5-star
lstm_145 = report[report.model_name == "LSTMB16_145"].copy().reset_index()
# 1-star, 3-star, 4-star, and 5-star
lstm_1345 = report[report.model_name == "LSTMB16_1345"].copy().reset_index()
# all reviews
lstm_all = report[(report.model_name == "LSTMB16") & (report.train_examples == 746766)].copy().reset_index()

data_file = lstm_15.file.values[0]

# Fix Classification Report For Various Runs

Have to remap classes from classification to the real class labels

In [ ]:
# map 2->5
cr15 = json.loads(lstm_15.classification_report.values[0])
cr15["5"] = cr15.pop("2")
lstm_15.loc[lstm_15.index, "classification_report"] = json.dumps(cr15)
lstm_15["label"] = "1,5 Star"
lstm_15 = ru.convert_dnn_report_format(lstm_15)

# map 3->5, 2->4
cr145 = json.loads(lstm_145.classification_report.values[0])
cr145["5"] = cr145.pop("3")
cr145["4"] = cr145.pop("2")
lstm_145.loc[lstm_145.index, "classification_report"] = json.dumps(cr145)
lstm_145["label"] = "1,4,5 Star"
lstm_145 = ru.convert_dnn_report_format(lstm_145)

# map 4->5, 3->4, 2->3
cr1345 = json.loads(lstm_1345.classification_report.values[0])
cr1345["5"] = cr1345.pop("4")
cr1345["4"] = cr1345.pop("3")
cr1345["3"] = cr1345.pop("2")
lstm_1345.loc[lstm_1345.index, "classification_report"] = json.dumps(cr1345)
lstm_1345["label"] = "1,3,4,5 Star"
lstm_1345 = ru.convert_dnn_report_format(lstm_1345)


lstm_all["label"] = "All"
lstm_all = ru.convert_dnn_report_format(lstm_all)

## Calculate Scores For Our Models

In [ ]:
lstm_15["eval_metric"] = 2 / (1 / lstm_15["1_precision"] + 1 / lstm_15["5_precision"])
lstm_145["eval_metric"] = 3 / (1 / lstm_145["1_precision"] + 1 / lstm_145["4_recall"] + 1 / lstm_145["5_precision"])
lstm_1345["eval_metric"] = 4 / (1 / lstm_1345["1_precision"] + 
                               1 / lstm_1345["3_recall"] + 
                               1 / lstm_1345["4_recall"] + 
                               1 / lstm_1345["5_precision"])
lstm_all["eval_metric"] = 5 / (1 / lstm_all["1_precision"] + 
                               1 / lstm_all["2_recall"] + 
                               1 / lstm_all["3_recall"] + 
                               1 / lstm_all["4_recall"] + 
                               1 / lstm_all["5_precision"])

In [ ]:
report_all = pd.DataFrame()
report_all = report_all.append(lstm_15, sort=False)
report_all = report_all.append(lstm_145, sort=False)
report_all = report_all.append(lstm_1345, sort=False)
report_all = report_all.append(lstm_all, sort=False)

# Print Model Scores

In [ ]:
f, a = plt.subplots(1, 1, figsize=(20, 5))
_ = sns.lineplot(report_all.label, report_all.eval_metric, marker="o", sort=False)
_ = a.set_title("Model Score")
_ = a.set_xlabel("Star Ratings")
_ = a.set_ylabel("Score")

# Classification Report

* Recall stays roughly the same across
* Precision drops as we introduce more classes
  * 1-star and 3-star review precision drops pretty quickly as we introduct more classes

In [ ]:
f, a = plt.subplots(1, 2, figsize=(20, 5), sharey=True)
sns.lineplot(report_all.label, report_all["1_recall"], sort=False, label="1-star", marker="o", ax=a[0])
sns.lineplot(report_all.label, report_all["2_recall"], sort=False, label="2-star", marker="o", ax=a[0])
sns.lineplot(report_all.label, report_all["3_recall"], sort=False, label="3-star", marker="o", ax=a[0])
sns.lineplot(report_all.label, report_all["4_recall"], sort=False, label="4-star", marker="o", ax=a[0])
sns.lineplot(report_all.label, report_all["5_recall"], sort=False, label="5-star", marker="o", ax=a[0])
_ = a[0].set_xlabel("Star Ratings")
_ = a[0].set_ylabel("")
_ = a[0].set_title("Recall")

sns.lineplot(report_all.label, report_all["1_precision"], sort=False, label="1-star", marker="o", ax=a[1])
sns.lineplot(report_all.label, report_all["2_precision"], sort=False, label="2-star", marker="o", ax=a[1])
sns.lineplot(report_all.label, report_all["3_precision"], sort=False, label="3-star", marker="o", ax=a[1])
sns.lineplot(report_all.label, report_all["4_precision"], sort=False, label="4-star", marker="o", ax=a[1])
sns.lineplot(report_all.label, report_all["5_precision"], sort=False, label="5-star", marker="o", ax=a[1])
_ = a[1].set_xlabel("Star Ratings")
_ = a[1].set_ylabel("")
_ = a[1].set_title("Precision")

# Confusion Matrix For Various Runs

## 1-star and 5-star Only

* This is our baseline. We do have some 1-star misclassified as 5-star and vice versa but numbers look pretty reasonable. < 10% for both classes.

In [ ]:
pd.DataFrame(json.loads(lstm_15.confusion_matrix.values[0])).rename({0: 1, 1: 5}, axis=1)

## 1-star, 4-star, 5-star

* 1-star most likely misclassified as 5-star
* 4-star most likely misclassified as 5-star
* 5-star most likely misclassified as 4-star

In [ ]:
pd.DataFrame(json.loads(lstm_145.confusion_matrix.values[0])).rename({0: 1, 2: 5, 1: 4}, axis=1)

## 1-star, 3-star, 4-star, 5-star

* 1-star most likely misclassified as 3-star
* <b>3-star most likely misclassified as 4-star (but 1 and 5 are pretty close)</b>
* 4-star most likely misclassified as 5-star
* 5-star most likely misclassified as 4-star

In [ ]:
pd.DataFrame(json.loads(lstm_1345.confusion_matrix.values[0])).rename({0: 1, 1: 3, 2: 4, 3: 5}, axis=1)

## All star ratings

* 1-star most likely misclassified as 3-star
* <b>2-star most likely misclassified as 1-star, 2-star is more likely to be misclassified as 1-star than itself</b>
* <b>3-star most likley misclassified as 5-star, then 4-star and 1-star</b>
* 4-star most likely misclassified as 5-star
* 5-star most likely misclassified as 4-star

In [ ]:
pd.DataFrame(json.loads(lstm_all.confusion_matrix.values[0])).rename({0: 1, 1: 2, 2: 3, 3: 4, 4: 5}, axis=1)

# Look at Specific Reviews That Are Misclassified

In [ ]:
# Read data file - we know they all started with the same data file so let's just pick one to start with
DATA_FILE = lstm_15.file.values[0]
print(f'Reading datafile: {DATA_FILE}')
df = pd.read_csv(DATA_FILE, encoding='utf8', engine='python')

# 1-Star and 5-Star Only

In [ ]:
MAX_SEQUENCE_LENGTH = 100

def load_and_predict(report: pd.DataFrame, reviews_train, reviews_test, y_train, y_test):
  """
  :param report: report DF
  :param classes: list of classes to keep
  """
  print(f'Loading tokenizer file: {report.tokenizer_file.values[0]}')
  with open(report.tokenizer_file.values[0], 'rb') as file:
    t = pickle.load(file)

  # Pre-process our features (review body)
  # t = Tokenizer(oov_token="<UNK>")
  # fit the tokenizer on the documents
  # t.fit_on_texts(reviews_train)
  # tokenize both our training and test data
  train_sequences = t.texts_to_sequences(reviews_train)
  test_sequences = t.texts_to_sequences(reviews_test)

  print("Vocabulary size={}".format(len(t.word_counts)))
  print("Number of Documents={}".format(t.document_count))


  # pad our reviews to the max sequence length
  X_train = sequence.pad_sequences(train_sequences, maxlen=MAX_SEQUENCE_LENGTH)
  X_test = sequence.pad_sequences(test_sequences, maxlen=MAX_SEQUENCE_LENGTH)

  print('Train review vectors shape:', X_train.shape, ' Test review vectors shape:', X_test.shape)

  print(f'Loading model file: {report.model_file.values[0]}')
  model = load_model(report.model_file.values[0])
  print(f'Predicting test set...')
  y_predict = model.predict(X_test)

  print(f'Unencoding predictions...')
  y_predict_unencoded = ku.unencode(y_predict)

  return model, y_predict, y_predict_unencoded




### Incorrect Results For 1-Star and 5-Star only

* Unmap predictions to original star ratings
* merge with reviews so we can look at the text

In [ ]:
# predict 15
df_15 = df[(df[LABEL_COLUMN] == 1) | (df[LABEL_COLUMN] == 5)]
# have to set 5's to 2's so we can compute class weights later
df_15.loc[df[LABEL_COLUMN] == 5, LABEL_COLUMN] = 2
CLASSES = 2

ratings_15 = df_15[LABEL_COLUMN]
reviews_15 = df_15[REVIEW_COLUMN]

reviews_train_15, reviews_test_15, y_train_15, y_test_15 = train_test_split(reviews_15, ratings_15, random_state=1)

model_15, y_predict_15, y_predict_unencoded_15 = load_and_predict(lstm_15, reviews_train_15, reviews_test_15, y_train_15, y_test_15)

In [ ]:
results_15 = pd.DataFrame({"true_15": y_test_15, "predict_15": y_predict_unencoded_15})

# unmap preidctions 2->
results_15.loc[results_15.true_15 == 2, "true_15"] = 5
results_15.loc[results_15.predict_15 == 2, "predict_15"] = 5

results_15 = results_15.join(reviews_test_15).reset_index().rename({"index": "index_orig"}, axis=1)


In [ ]:
results_15[results_15.true_15 != results_15.predict_15].head(10)

# 1-Star, 4-Star, and 5-Star

In [ ]:
# predict 145
df_145 = df[(df[LABEL_COLUMN] == 1) | (df[LABEL_COLUMN] == 4) | (df[LABEL_COLUMN] == 5)]
# have to set 4->2, 5->3
df_145.loc[df[LABEL_COLUMN] == 4, LABEL_COLUMN] = 2
df_145.loc[df[LABEL_COLUMN] == 5, LABEL_COLUMN] = 3

ratings_145 = df_145[LABEL_COLUMN]
reviews_145 = df_145[REVIEW_COLUMN]

reviews_train_145, reviews_test_145, y_train_145, y_test_145 = train_test_split(reviews_145, ratings_145, random_state=1)

model_145, y_predict_145, y_predict_unencoded_145 = load_and_predict(lstm_145, 
                                                                     reviews_train_145, 
                                                                     reviews_test_145, 
                                                                     y_train_145, 
                                                                     y_test_145)



In [ ]:
results_145 = pd.DataFrame({"true_145": y_test_145, "predict_145": y_predict_unencoded_145})

# unmap preidctions 2->4, 3->5
results_145.loc[results_145.true_145 == 2, "true_145"] = 4
results_145.loc[results_145.true_145 == 3, "true_145"] = 5
results_145.loc[results_145.predict_145 == 2, "predict_145"] = 4
results_145.loc[results_145.predict_145 == 3, "predict_145"] = 5

results_145.head()

In [ ]:
results_145 = results_145.join(reviews_test_145).reset_index().rename({"index": "index_orig"}, axis=1)
results_145.head()

### Misclassified 1-Star Reviews

In [ ]:
results_145[(results_145.true_145 == 1) & 
            (results_145.true_145 != results_145.predict_145)].head(10)

### Misclassified 4-Star Reviews

* 4 misclassified as 5

In [ ]:
results_145[(results_145.true_145 == 4) & 
            (results_145.predict_145 == 5) & 
            (results_145.true_145 != results_145.predict_145)].head(10)

* 4 misclassified as 1

In [ ]:
results_145[(results_145.true_145 == 4) & 
            (results_145.predict_145 == 1) & 
            (results_145.true_145 != results_145.predict_145)].head(10)

# 1-Star, 3-Star, 4-Star, 5-Star Ratings

In [ ]:
# predict 1345
df_1345 = df[(df[LABEL_COLUMN] == 1) | (df[LABEL_COLUMN] == 3) | (df[LABEL_COLUMN] == 4) | (df[LABEL_COLUMN] == 5)]
# have to set 3->2, 4->3, 5->4
df_1345.loc[df[LABEL_COLUMN] == 3, LABEL_COLUMN] = 2
df_1345.loc[df[LABEL_COLUMN] == 4, LABEL_COLUMN] = 3
df_1345.loc[df[LABEL_COLUMN] == 5, LABEL_COLUMN] = 4

ratings_1345 = df_1345[LABEL_COLUMN]
reviews_1345 = df_1345[REVIEW_COLUMN]

reviews_train_1345, reviews_test_1345, y_train_1345, y_test_1345 = train_test_split(reviews_1345, ratings_1345, random_state=1)

model_1345, y_predict_1345, y_predict_unencoded_1345 = load_and_predict(lstm_1345, 
                                                                     reviews_train_1345, 
                                                                     reviews_test_1345, 
                                                                     y_train_1345, 
                                                                     y_test_1345)



In [ ]:
results_1345 = pd.DataFrame({"true_1345": y_test_1345, "predict_1345": y_predict_unencoded_1345})

# unmap preidctions 2->3, 3->4, 4->5
results_1345.loc[results_1345.true_1345 == 4, "true_1345"] = 5
results_1345.loc[results_1345.true_1345 == 3, "true_1345"] = 4
results_1345.loc[results_1345.true_1345 == 2, "true_1345"] = 3
results_1345.loc[results_1345.predict_1345 == 4, "predict_1345"] = 5
results_1345.loc[results_1345.predict_1345 == 3, "predict_1345"] = 4
results_1345.loc[results_1345.predict_1345 == 2, "predict_1345"] = 3

results_1345 = results_1345.join(reviews_test_1345).reset_index().rename({"index": "index_orig"}, axis=1)

### Misclassified 3-Star Reviews

* 3 as 4

In [ ]:
results_1345[(results_1345.true_1345 == 3) & 
            (results_1345.predict_1345 == 4) & 
             (results_1345.true_1345 != results_1345.predict_1345)].head(10)

* 3 as 1

In [ ]:
results_1345[(results_1345.true_1345 == 3) & 
            (results_1345.predict_1345 == 1) & 
             (results_1345.true_1345 != results_1345.predict_1345)].head(10)

# All Star Ratings

In [ ]:
# # predict 1345
# df_1345 = df[(df[LABEL_COLUMN] == 1) | (df[LABEL_COLUMN] == 3) | (df[LABEL_COLUMN] == 4) | (df[LABEL_COLUMN] == 5)]
# # have to set 3->2, 4->3, 5->4
# df_1345.loc[df[LABEL_COLUMN] == 3, LABEL_COLUMN] = 2
# df_1345.loc[df[LABEL_COLUMN] == 4, LABEL_COLUMN] = 3
# df_1345.loc[df[LABEL_COLUMN] == 5, LABEL_COLUMN] = 4

ratings_all = df_1345[LABEL_COLUMN]
reviews_all = df_1345[REVIEW_COLUMN]

reviews_train_all, reviews_test_all, y_train_all, y_test_all = train_test_split(reviews_all, ratings_all, random_state=1)

model_all, y_predict_all, y_predict_unencoded_all = load_and_predict(lstm_all, 
                                                                     reviews_train_all, 
                                                                     reviews_test_all, 
                                                                     y_train_all, 
                                                                     y_test_all)



In [ ]:
results_all = pd.DataFrame({"true_all": y_test_all, "predict_all": y_predict_unencoded_all})

# don't have to unmap predictions

results_all = results_all.join(reviews_test_all).reset_index().rename({"index": "index_orig"}, axis=1)

### Misclassified Samples

* 1 as 5

In [ ]:
results_all[(results_all.true_all == 1) & 
            (results_all.predict_all == 5) & 
             (results_all.true_all != results_all.predict_all)].head(10)

# Join all Results Dataframe Into One

In [ ]:
results_comprehensive = pd.merge(pd.merge(pd.merge(results_15.drop("review_body", axis=1), 
                                                results_145.drop("review_body", axis=1), how='outer', 
                                                left_on="index_orig", right_on="index_orig"),
                                      results_1345.drop("review_body", axis=1), how='outer', 
                                                left_on="index_orig", right_on="index_orig"),
                                      results_all, how='outer', 
                                                left_on="index_orig", right_on="index_orig")

results_comprehensive = results_comprehensive.dropna(subset=["review_body"])
results_comprehensive.head()



In [ ]:
print(datetime.now())